In [1]:
# Import packages for data manipulation
import numpy as np
import pandas as pd

# Import packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import packages for statistical analysis/hypothesis testing
from scipy import stats

# Download / prepare Liiga dataset (if not already in memory)
import json
import urllib.request

url = "https://liiga.fi/api/v2/games?season=2025"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as response:
    data = json.loads(response.read().decode())

df = pd.json_normalize(data)
cols = [
    "id",
    "homeTeam.teamName",
    "awayTeam.teamName",
    "homeTeam.goals",
    "awayTeam.goals",
    "homeTeam.expectedGoals",
    "awayTeam.expectedGoals",
    "spectators",
]
df_liiga = df[cols].copy()
df_liiga.columns = [
    "id",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "spectators",
]
df_liiga["home_win"] = (
    df_liiga["home_goals"] > df_liiga["away_goals"]
).astype(int)

In [2]:
# 1. Display summary statistics
print(df_liiga.describe())

# 2. Check and handle missing values in xG columns
print("\nMissing values before cleaning:")
print(df_liiga[["home_xg", "away_xg"]].isna().sum())

# Drop rows with missing xG for hypothesis testing
df_clean = df_liiga.dropna(subset=["home_xg", "away_xg"]).copy()

# 3. Compute group means of home_xg by match outcome
print("\nMean home_xg by match outcome:")
print(df_clean.groupby("home_win")["home_xg"].mean())

                 id  home_goals  away_goals     home_xg     away_xg  \
count    614.000000  614.000000  614.000000  582.000000  582.000000   
mean    5817.799674    2.952769    2.710098    2.770155    2.522027   
std    15878.503914    1.718164    1.659614    0.874530    0.850825   
min        1.000000    0.000000    0.000000    0.530000    0.340000   
25%      154.250000    2.000000    2.000000    2.130000    1.940000   
50%      307.500000    3.000000    3.000000    2.770000    2.440000   
75%      460.750000    4.000000    4.000000    3.367500    3.060000   
max    52454.000000    9.000000    9.000000    5.840000    6.320000   

         spectators    home_win  
count    614.000000  614.000000  
mean    4385.628664    0.532573  
std     2270.005240    0.499345  
min        0.000000    0.000000  
25%     3015.500000    0.000000  
50%     4013.500000    1.000000  
75%     5147.500000    1.000000  
max    12700.000000    1.000000  

Missing values before cleaning:
home_xg    32
away_xg

In [3]:
# 1. Isolate the two sample groups
home_win_xg = df_clean[df_clean["home_win"] == 1]["home_xg"]
home_loss_xg = df_clean[df_clean["home_win"] == 0]["home_xg"]

# 2. Conduct two-sample Welch's t-test
t_stat, p_val = stats.ttest_ind(
    a=home_win_xg, b=home_loss_xg, equal_var=False
)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")

# 3. Decision rule
alpha = 0.05
if p_val < alpha:
    print("\nResult: Reject the null hypothesis (H0).")
    print(
        "There is a statistically significant difference in home xG between wins and losses."
    )
else:
    print("\nResult: Fail to reject the null hypothesis (H0).")

T-statistic: 3.0770
P-value: 2.1965e-03

Result: Reject the null hypothesis (H0).
There is a statistically significant difference in home xG between wins and losses.
